# Task 2: Named Entity Recognition (CoNLL-2003)
**CENG 467 — NLU&G Take-Home Midterm | Student ID: 300201071**

Models: BiLSTM-CRF, BERT-NER

Metrics: Precision, Recall, F1-score (entity-level)

In [1]:
!pip install -q "datasets<3.0.0" datasets transformers seqeval pytorch-crf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import os, random, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
from seqeval.metrics import classification_report as seq_report, precision_score as seq_p, recall_score as seq_r, f1_score as seq_f1

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

NER_TAGS = ['O','B-PER','I-PER','B-ORG','I-ORG','B-LOC','I-LOC','B-MISC','I-MISC']
TAG2IDX = {t:i for i,t in enumerate(NER_TAGS)}
IDX2TAG = {i:t for t,i in TAG2IDX.items()}

Device: cuda


## 1. Load CoNLL-2003

In [3]:
dataset = load_dataset('conll2003')
print(f'Train: {len(dataset["train"])}, Val: {len(dataset["validation"])}, Test: {len(dataset["test"])}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Train: 14041, Val: 3250, Test: 3453


## 2. Model A: BiLSTM-CRF

In [4]:
# Build vocab
counter = Counter()
for ex in dataset['train']:
    for t in ex['tokens']: counter[t.lower()] += 1
vocab = {'<pad>':0, '<unk>':1}
for w,c in counter.most_common():
    if c>=1: vocab[w] = len(vocab)
print(f'Vocab: {len(vocab)}')

MAX_LEN = 128
def prep(split):
    tids, tags, lens = [],[],[]
    for ex in split:
        tk = ex['tokens']; tg = ex['ner_tags']
        ti = [vocab.get(t.lower(), 1) for t in tk[:MAX_LEN]]
        ta = [tg[i] for i in range(min(len(tg), MAX_LEN))]
        l = len(ti)
        ti += [0]*(MAX_LEN-l); ta += [0]*(MAX_LEN-l)
        tids.append(ti); tags.append(ta); lens.append(l)
    return np.array(tids,dtype=np.int64), np.array(tags,dtype=np.int64), np.array(lens,dtype=np.int64)

Xtr,Ytr,Ltr = prep(dataset['train'])
Xv,Yv,Lv = prep(dataset['validation'])
Xt,Yt,Lt = prep(dataset['test'])

class NDS(Dataset):
    def __init__(s,x,y,l): s.x,s.y,s.l = torch.LongTensor(x),torch.LongTensor(y),torch.LongTensor(l)
    def __len__(s): return len(s.l)
    def __getitem__(s,i): return s.x[i],s.y[i],s.l[i]

BS=32
tr_dl=DataLoader(NDS(Xtr,Ytr,Ltr),batch_size=BS,shuffle=True)
vl_dl=DataLoader(NDS(Xv,Yv,Lv),batch_size=BS)
ts_dl=DataLoader(NDS(Xt,Yt,Lt),batch_size=BS)

Vocab: 21011


In [5]:
class BiLSTMCRF(nn.Module):
    def __init__(self, vs, ed=100, hd=256, nt=9, do=0.5):
        super().__init__()
        self.emb = nn.Embedding(vs, ed, padding_idx=0)
        self.lstm = nn.LSTM(ed, hd, batch_first=True, bidirectional=True, num_layers=2, dropout=do)
        self.drop = nn.Dropout(do)
        self.h2t = nn.Linear(hd*2, nt)
        self.crf = CRF(nt, batch_first=True)
    def _emit(self, x):
        e = self.drop(self.emb(x))
        o,_ = self.lstm(e)
        return self.h2t(self.drop(o))
    def forward(self, x, tags, mask):
        return -self.crf(self._emit(x), tags, mask=mask, reduction='mean')
    def decode(self, x, mask):
        return self.crf.decode(self._emit(x), mask=mask)

model = BiLSTMCRF(len(vocab)).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print(f'BiLSTM-CRF params: {sum(p.numel() for p in model.parameters()):,}')

for ep in range(15):
    model.train(); tl=0
    for xi,yi,li in tqdm(tr_dl, desc=f'Ep {ep+1}/15', leave=False):
        xi,yi,li = xi.to(DEVICE),yi.to(DEVICE),li.to(DEVICE)
        mask = torch.arange(MAX_LEN,device=DEVICE).unsqueeze(0) < li.unsqueeze(1)
        opt.zero_grad(); loss=model(xi,yi,mask); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step(); tl+=loss.item()
    model.eval(); vl=0
    with torch.no_grad():
        for xi,yi,li in vl_dl:
            xi,yi,li=xi.to(DEVICE),yi.to(DEVICE),li.to(DEVICE)
            mask=torch.arange(MAX_LEN,device=DEVICE).unsqueeze(0)<li.unsqueeze(1)
            vl+=model(xi,yi,mask).item()
    print(f'  Ep {ep+1}: Train={tl/len(tr_dl):.4f} Val={vl/len(vl_dl):.4f}')

# Predict
model.eval(); crf_true,crf_pred=[],[]
with torch.no_grad():
    for xi,yi,li in ts_dl:
        xi,li=xi.to(DEVICE),li.to(DEVICE)
        mask=torch.arange(MAX_LEN,device=DEVICE).unsqueeze(0)<li.unsqueeze(1)
        dec=model.decode(xi,mask)
        for i,(seq,l) in enumerate(zip(dec,li)):
            ln=l.item()
            crf_pred.append([IDX2TAG[t] for t in seq[:ln]])
            crf_true.append([IDX2TAG[yi[i][j].item()] for j in range(ln)])

crf_p=seq_p(crf_true,crf_pred); crf_r=seq_r(crf_true,crf_pred); crf_f=seq_f1(crf_true,crf_pred)
print(f'\nBiLSTM-CRF TEST => P={crf_p:.4f} R={crf_r:.4f} F1={crf_f:.4f}')
print(seq_report(crf_true,crf_pred))

BiLSTM-CRF params: 4,415,960


Ep 1/15:   0%|          | 0/439 [00:00<?, ?it/s]

  Ep 1: Train=7.9706 Val=5.6360


Ep 2/15:   0%|          | 0/439 [00:00<?, ?it/s]

  Ep 2: Train=4.9633 Val=4.1872


...

  Ep 13: Train=1.4138 Val=2.1623


Ep 14/15:   0%|          | 0/439 [00:00<?, ?it/s]

  Ep 14: Train=1.3155 Val=2.1425


Ep 15/15:   0%|          | 0/439 [00:00<?, ?it/s]

  Ep 15: Train=1.2344 Val=2.1031

BiLSTM-CRF TEST => P=0.8019 R=0.7045 F1=0.7500
              precision    recall  f1-score   support

         LOC       0.86      0.78      0.82      1668
        MISC       0.73      0.65      0.69       702
         ORG       0.77      0.64      0.70      1661
         PER       0.80      0.72      0.76      1617

   micro avg       0.80      0.70      0.75      5648
   macro avg       0.79      0.70      0.74      5648
weighted avg       0.80      0.70      0.75      5648



## 3. Model B: BERT-NER

In [6]:
btok = AutoTokenizer.from_pretrained('bert-base-cased')
bmodel = AutoModelForTokenClassification.from_pretrained('bert-base-cased', num_labels=len(NER_TAGS))

def align_and_tokenize(split):
    all_enc = {'input_ids':[], 'attention_mask':[], 'labels':[]}
    for ex in split:
        tokens, tags = ex['tokens'], ex['ner_tags']
        enc = btok(tokens, is_split_into_words=True, truncation=True, padding='max_length', max_length=128)
        wids = enc.word_ids(); labels=[]; prev=None
        for wid in wids:
            if wid is None: labels.append(-100)
            elif wid!=prev: labels.append(tags[wid])
            else: labels.append(-100)
            prev=wid
        all_enc['input_ids'].append(enc['input_ids'])
        all_enc['attention_mask'].append(enc['attention_mask'])
        all_enc['labels'].append(labels)
    ds = HFDataset.from_dict(all_enc); ds.set_format('torch')
    return ds

def ner_metrics(ep):
    logits,labels=ep; preds=np.argmax(logits,axis=2)
    tt,tp=[],[]
    for i in range(len(labels)):
        ts,ps=[],[]
        for j in range(len(labels[i])):
            if labels[i][j]!=-100:
                ts.append(IDX2TAG[labels[i][j]]); ps.append(IDX2TAG[preds[i][j]])
        tt.append(ts); tp.append(ps)
    return {'precision':seq_p(tt,tp),'recall':seq_r(tt,tp),'f1':seq_f1(tt,tp)}

tr_ds=align_and_tokenize(dataset['train'])
vl_ds=align_and_tokenize(dataset['validation'])
ts_ds=align_and_tokenize(dataset['test'])

args = TrainingArguments(
    output_dir='/content/bert_ner', num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    learning_rate=3e-5, weight_decay=0.01, eval_strategy='epoch',
    save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1', seed=SEED, logging_steps=200,
    report_to='none', fp16=torch.cuda.is_available())

trainer = Trainer(model=bmodel, args=args, train_dataset=tr_ds,
                  eval_dataset=vl_ds, data_collator=DataCollatorForTokenClassification(btok),
                  compute_metrics=ner_metrics)
trainer.train()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2634, training_loss=0.04656545228367967, metrics={'train_runtime': 374.2125, 'train_samples_per_second': 112.564, 'train_steps_per_second': 7.039, 'total_flos': 2751824963545344.0, 'train_loss': 0.04656545228367967, 'epoch': 3.0})

In [7]:
res = trainer.predict(ts_ds)
logits,labels = res.predictions, res.label_ids
preds = np.argmax(logits, axis=2)
bert_true,bert_pred=[],[]
for i in range(len(labels)):
    ts,ps=[],[]
    for j in range(len(labels[i])):
        if labels[i][j]!=-100:
            ts.append(IDX2TAG[labels[i][j]]); ps.append(IDX2TAG[preds[i][j]])
    bert_true.append(ts); bert_pred.append(ps)

bert_p=seq_p(bert_true,bert_pred); bert_r=seq_r(bert_true,bert_pred); bert_f=seq_f1(bert_true,bert_pred)
print(f'BERT-NER TEST => P={bert_p:.4f} R={bert_r:.4f} F1={bert_f:.4f}')
print(seq_report(bert_true,bert_pred))

BERT-NER TEST => P=0.9102 R=0.9233 F1=0.9167
              precision    recall  f1-score   support

         LOC       0.93      0.94      0.93      1666
        MISC       0.78      0.84      0.80       702
         ORG       0.90      0.91      0.91      1661
         PER       0.97      0.96      0.96      1615

   micro avg       0.91      0.92      0.92      5644
   macro avg       0.89      0.91      0.90      5644
weighted avg       0.91      0.92      0.92      5644



## 4. Error Analysis

In [8]:
def analyze_errors(true_tags, pred_tags, name):
    boundary,confusion,missed,spurious = 0,0,0,0
    examples = {'boundary':[],'confusion':[],'missed':[],'spurious':[]}
    for i in range(len(true_tags)):
        for j in range(len(true_tags[i])):
            t,p = true_tags[i][j], pred_tags[i][j]
            if t!=p:
                tok = dataset['test'][i]['tokens'][j] if j<len(dataset['test'][i]['tokens']) else '?'
                info = f'{tok}: true={t} pred={p}'
                if t=='O' and p!='O': spurious+=1; examples['spurious'].append(info)
                elif t!='O' and p=='O': missed+=1; examples['missed'].append(info)
                elif t!='O' and p!='O' and t.split('-')[-1]!=p.split('-')[-1]: confusion+=1; examples['confusion'].append(info)
                elif t!='O' and p!='O': boundary+=1; examples['boundary'].append(info)
    print(f'\n{name} Error Analysis:')
    print(f'  Boundary: {boundary}, Type Confusion: {confusion}, Missed: {missed}, Spurious: {spurious}')
    for k,v in examples.items():
        if v: print(f'  Sample {k}: {v[:3]}')

analyze_errors(crf_true, crf_pred, 'BiLSTM-CRF')
analyze_errors(bert_true, bert_pred, 'BERT-NER')


BiLSTM-CRF Error Analysis:
  Boundary: 98, Type Confusion: 537, Missed: 1566, Spurious: 639
  Sample boundary: ['World: true=I-MISC pred=B-MISC', 'F.A.: true=I-MISC pred=B-MISC', 'da: true=I-PER pred=B-PER']
  Sample confusion: ['CHINA: true=B-PER pred=B-LOC', 'Japan: true=B-LOC pred=B-ORG', 'Syria: true=B-LOC pred=B-ORG']
  Sample missed: ['Nadim: true=B-PER pred=O', 'Ladki: true=I-PER pred=O', 'Emirates: true=I-LOC pred=O']
  Sample spurious: ['rarely: true=O pred=B-PER', 'fine: true=O pred=B-PER', '2002: true=O pred=B-LOC']

BERT-NER Error Analysis:
  Boundary: 31, Type Confusion: 381, Missed: 129, Spurious: 243
  Sample boundary: ['World: true=I-MISC pred=B-MISC', 'F.A.: true=I-MISC pred=B-MISC', 'F.A.: true=I-MISC pred=B-MISC']
  Sample confusion: ['JAPAN: true=B-LOC pred=B-ORG', 'CHINA: true=B-PER pred=B-ORG', 'Bitar: true=B-PER pred=B-ORG']
  Sample missed: ['ITALY: true=B-LOC pred=O', '1995: true=B-MISC pred=O', 'SKIING-WORLD: true=B-MISC pred=O']
  Sample spurious: ['LUCKY: t

## 5. Final Results Summary (for LaTeX Report)

In [9]:
print('\n' + '='*60)
print('  TASK 2 — FINAL RESULTS (Copy to LaTeX)')
print('='*60)
print(f'{"Model":<15} {"Precision":<12} {"Recall":<12} {"F1-score":<12}')
print('-'*51)
print(f'{"BiLSTM-CRF":<15} {crf_p:<12.4f} {crf_r:<12.4f} {crf_f:<12.4f}')
print(f'{"BERT-NER":<15} {bert_p:<12.4f} {bert_r:<12.4f} {bert_f:<12.4f}')
print('-'*51)


  TASK 2 — FINAL RESULTS (Copy to LaTeX)
Model           Precision    Recall       F1-score    
---------------------------------------------------
BiLSTM-CRF      0.8019       0.7045       0.7500      
BERT-NER        0.9102       0.9233       0.9167      
---------------------------------------------------
